# 02 · Parse, validate and load into SQLite

Phases 2–3: flatten JSON into rows, check keys and referential integrity, create the 7-table schema and load it.

**Database:** SQLite (`data/nhl.db`) · **connector:** Python's built-in `sqlite3`.

Every step below is also wrapped in `nhl_pipeline.pipeline.build_database()` — the last cell shows the one-line equivalent (`python run_pipeline.py`).

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nhl_pipeline import config, parsers
from nhl_pipeline.api_client import NHLClient
from nhl_pipeline.demo import DemoClient

# "api"  = live NHL public API (no key needed)
# "demo" = offline synthetic season with the same JSON shape (no network)
SOURCE = "api"

client = DemoClient() if SOURCE == "demo" else NHLClient()
pd.set_option("display.max_columns", 40)
print("Source:", SOURCE, "| season:", config.SEASON, "| base URL:", config.BASE_URL)

from nhl_pipeline import datasets
from nhl_pipeline.db import connect, create_schema, insert_rows, split_by_fk, lookup, table_counts

MAX_BOXSCORES = None   # e.g. 50 for a quick test run of the box-score fallback
DB_PATH = config.DB_PATH

## Create the schema
The DDL lives in `sql/schema.sql` (PRIMARY KEY, FOREIGN KEY, NOT NULL, UNIQUE, indexes). Re-running it drops and recreates every table, so this notebook always does a clean load.

In [ ]:
conn = connect(DB_PATH)
create_schema(conn)
print(open(config.SCHEMA_PATH).read()[:1400], "...")

In [ ]:
load_log = []
def load(table, rows, fk=None):
    """Referential-integrity gate, then INSERT OR IGNORE. Logs what happened."""
    dropped = 0
    if fk:
        rows, bad = split_by_fk(rows, fk)
        dropped = len(bad)
    inserted = insert_rows(conn, table, rows)
    load_log.append({"table": table, "offered": len(rows) + dropped, "inserted": inserted,
                     "duplicates_ignored": len(rows) - inserted, "orphans_dropped": dropped})
    return inserted

## Tables 1–2 · teams and standings

In [ ]:
standings_json = client.standings()
team_rows = parsers.parse_teams(standings_json)
load("teams", team_rows)
team_ids = lookup(conn, "SELECT team_abbrev, team_id FROM teams")   # generated ids
valid_teams = set(team_ids.values())
load("standings", parsers.parse_standings(standings_json, team_ids, config.SEASON), fk={"team_id": valid_teams})
pd.read_sql("SELECT * FROM teams LIMIT 5", conn)

In [ ]:
pd.read_sql("""SELECT t.team_abbrev, s.* FROM standings s JOIN teams t USING (team_id)
               ORDER BY points DESC LIMIT 5""", conn)

## Table 3 · players
One roster call per team; forwards, defensemen and goalies flattened into one list.

In [ ]:
for abbrev in sorted(team_ids):
    load("players", parsers.parse_roster(client.roster(abbrev), team_ids[abbrev]), fk={"team_id": valid_teams})
player_team = lookup(conn, "SELECT player_id, team_id FROM players")
player_pos = lookup(conn, "SELECT player_id, position FROM players")
valid_players = set(player_team)
pd.read_sql("SELECT position, COUNT(*) AS players FROM players GROUP BY position", conn)

## Table 4 · games
Called once per team, so every game arrives twice; `INSERT OR IGNORE` on `game_id` keeps one copy. Games against clubs outside `teams` are set aside as orphans.

In [ ]:
orphan_games = {}
for abbrev in sorted(team_ids):
    valid, orphans = parsers.parse_schedule(client.schedule(abbrev), team_ids, config.SEASON)
    orphan_games.update({o["game_id"]: o for o in orphans})
    load("games", valid)
print(len(orphan_games), "orphan games skipped (non-NHL opponent)")
pd.read_sql("SELECT game_state, game_type, COUNT(*) AS games FROM games GROUP BY 1, 2", conn)

## Table 5 · game_stats (skaters only)
Uses the provided dataset when it is in `data/raw/datasets/`; otherwise fetches box scores for completed games.

In [ ]:
fk_stats = {"player_id": valid_players, "team_id": valid_teams}
gs_file = datasets.find_dataset("game_stats")
if gs_file and SOURCE != "demo":
    print("Loading dataset", gs_file.name)
    load("game_stats", datasets.load_game_stats(gs_file, team_ids, player_team), fk=fk_stats)
else:
    ids = [r[0] for r in conn.execute(
        "SELECT game_id FROM games WHERE game_state IN ('OFF','FINAL') AND game_type = 2 ORDER BY game_date")]
    ids = ids[:MAX_BOXSCORES] if MAX_BOXSCORES else ids
    print(f"Fetching {len(ids)} box scores …")
    rows = []
    for i, gid in enumerate(ids, 1):
        rows.extend(parsers.parse_boxscore(client.boxscore(gid), team_ids))
        if i % 200 == 0:
            print(" ", i)
    load("game_stats", rows, fk=fk_stats)
pd.read_sql("SELECT * FROM game_stats LIMIT 5", conn)

## Tables 6–7 · skater and goalie season stats
Datasets when present, otherwise one club-stats call per team. Rows are routed by position (`G` → goalies).

In [ ]:
files = [f for f in (datasets.find_dataset("skater_season_stats"), datasets.find_dataset("goalie_season_stats")) if f]
if files and SOURCE != "demo":
    print("Loading datasets", [f.name for f in files])
    skaters, goalies = datasets.load_season_stats(files, team_ids, player_team, player_pos, config.SEASON)
    load("skater_season_stats", skaters, fk=fk_stats)
    load("goalie_season_stats", goalies, fk=fk_stats)
else:
    for abbrev in sorted(team_ids):
        skaters, goalies = parsers.parse_club_stats(client.club_stats(abbrev), team_ids[abbrev], config.SEASON)
        load("skater_season_stats", skaters, fk=fk_stats)
        load("goalie_season_stats", goalies, fk=fk_stats)
pd.read_sql("SELECT * FROM goalie_season_stats ORDER BY games_played DESC LIMIT 5", conn)

## Validation
What was inserted, what was ignored as a duplicate, what failed the foreign-key gate — then post-load checks that should all be 0.

In [ ]:
pd.DataFrame(load_log).groupby("table", sort=False).sum()

In [ ]:
from nhl_pipeline.pipeline import run_quality_checks
checks = run_quality_checks(conn)
print(json.dumps(checks, indent=2))
assert checks["foreign_key_violations"] == 0 and not checks["empty_tables"], "fix before using the DB"
table_counts(conn)

In [ ]:
conn.close()
# the dashboard shows the data source from this file
(DB_PATH.with_suffix(".run.json")).write_text(json.dumps(
    {"source": SOURCE, "season": config.SEASON, "built_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}))
print("Database ready:", DB_PATH)

### One-line equivalent
```bash
python run_pipeline.py            # live API
python run_pipeline.py --source demo
```
The script version builds into a temp file and swaps it in atomically, so a running dashboard never sees a half-loaded database.